# Анализ изображений · Часть 4: компьютерное зрение
**Практикум.** Лист с пятнами — что с ним, где пятна и сколько поражено? Считаем классикой и маской SAM,
дообучаем готовую сеть на пяти болезнях томата и экономим разметку активным обучением.

**Как работать.** Меню **«Среда выполнения → Выполнить все»** (Runtime → Run all) — и читайте сверху вниз.
В ячейках с полями справа меняйте значения и запускайте ячейку заново (▶ слева или Ctrl+Enter). Сломать ничего
нельзя: **«Среда выполнения → Отключиться и удалить среду»** вернёт всё как было. Код можно не читать — но если
интересно, он открыт и короткий.

Разделы 2 и 5 используют нейросети: на процессоре они работают, но медленнее. Если есть возможность, включите
видеокарту: **«Среда выполнения → Сменить среду → GPU»** (до запуска ячеек).

Снимки листьев — открытый набор **PlantVillage** (Mohanty, Hughes, Salathé, 2016), лицензия CC BY-SA 3.0, репозиторий [spMohanty/PlantVillage-Dataset](https://github.com/spMohanty/PlantVillage-Dataset).

**Номер участника.** Поставьте номер, который дал преподаватель, и запускайте ноутбук. У каждого номера своя
выборка данных; результаты вносите в общую таблицу группы — на разборе сравним, у кого что вышло и почему.

In [ ]:
#@title Ваш номер участника — выдаёт преподаватель
participant = 0  #@param {type:"slider", min:0, max:8, step:1}
print(f'Номер участника: {participant}. ' + ('Номер 0 — те же данные, что на слайдах.' if participant == 0 else
      'У вас своя выборка: числа будут отличаться от соседей — так и задумано, сравните их на разборе.'))

In [ ]:
#@title Подготовка: библиотеки и доступ к снимкам PlantVillage
import io, os, subprocess, urllib.parse, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/spMohanty/PlantVillage-Dataset/master/'
SAMPLES = {
    'apple_rust': 'raw/color/Apple___Cedar_apple_rust/46597071-9826-4f76-b853-083b76c5ef63___FREC_C.Rust 3666.JPG',
    'corn_gls': 'raw/color/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/8858dd29-eac0-443a-8213-425bbe223f7e___RS_GLSp 7334.JPG',
    'corn_healthy': 'raw/color/Corn_(maize)___healthy/3ac3f13d-352c-4390-bf06-94b5334cbfd8___R.S_HL 8206 copy 2.jpg',
    'grape_brot': 'raw/color/Grape___Black_rot/83b866ce-0785-4312-8e43-1ae8aa59d3bd___FAM_B.Rot 3518.JPG',
    'peach_bact': 'raw/color/Peach___Bacterial_spot/1df78012-4b87-40f4-bfa1-b868465c0b4d___Rut._Bact.S 1263.JPG',
    'soy_healthy': 'raw/color/Soybean___healthy/8f1ff646-db74-4924-abe8-373894711d26___RS_HL 5701.JPG',
    'tom_bact': 'raw/color/Tomato___Bacterial_spot/47798da0-ff59-497e-a2bb-e169c9138a7e___GCREC_Bact.Sp 3120.JPG',
    'tom_healthy': 'raw/color/Tomato___healthy/18fa8f7f-ef94-4a3b-9c79-5f4362323134___RS_HL 9977.JPG',
    'tom_late': 'raw/color/Tomato___Late_blight/6578c88a-fd14-450b-8e75-c94571decf83___GHLB2 Leaf 8849.JPG',
    'tom_mites': 'raw/color/Tomato___Spider_mites Two-spotted_spider_mite/2c2f7f98-8c5b-4981-bb74-f827c5f963fe___Com.G_SpM_FL 8734.JPG',
    'tom_mosaic': 'raw/color/Tomato___Tomato_mosaic_virus/6ebb483a-d1b6-4d7d-aad1-e717c2a36167___PSU_CG 2286.JPG',
    'tom_septoria': 'raw/color/Tomato___Septoria_leaf_spot/9acdb655-bca7-4223-a8e2-c201f6b3d188___Matt.S_CG 6834.JPG',
    'tom_target': 'raw/color/Tomato___Target_Spot/63a17b2f-817a-4717-9964-76ee7d6c3a63___Com.G_TgS_FL 9938.JPG',
    'tom_ylcv': 'raw/color/Tomato___Tomato_Yellow_Leaf_Curl_Virus/39b9e3b8-d3ba-4c0f-92be-e75ba728bce8___YLCV_GCREC 2711.JPG',
}

def get_image(path):
    """Скачать один снимок набора по его пути."""
    with urllib.request.urlopen(BASE + urllib.parse.quote(path), timeout=60) as r:
        return np.asarray(Image.open(io.BytesIO(r.read())).convert('RGB'))

def get_many(paths):
    """Скачать много снимков параллельно."""
    with ThreadPoolExecutor(16) as ex:
        return list(ex.map(get_image, paths))

def sample(name):
    return get_image(SAMPLES[name])

def list_dataset():
    """Имена всех снимков набора — без скачивания самих файлов (секунд 20 при первом вызове)."""
    if not os.path.exists('pv_list.txt'):
        if not os.path.exists('pv_repo'):
            subprocess.run(['git', 'clone', '-q', '--filter=blob:none', '--no-checkout', '--depth', '1',
                            'https://github.com/spMohanty/PlantVillage-Dataset', 'pv_repo'], check=True)
        out = subprocess.run(['git', '-C', 'pv_repo', 'ls-tree', '-r', '--name-only', 'HEAD', 'raw/color'],
                             capture_output=True, text=True, check=True).stdout
        with open('pv_list.txt', 'w') as f:
            f.write(out)
    names = open('pv_list.txt').read().splitlines()
    return [p for p in names if p.lower().endswith(('.jpg', '.jpeg', '.png'))]

def cls(path):
    return path.split('/')[2]

def show(images, titles=None, cols=6, size=2.3, colors=None):
    rows = (len(images) + cols - 1) // cols
    fig, axs = plt.subplots(rows, cols, figsize=(cols * size, rows * (size + 0.5)), squeeze=False)
    for ax in axs.ravel():
        ax.axis('off')
    for i, im in enumerate(images):
        ax = axs.ravel()[i]
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
        if titles:
            ax.set_title(titles[i], fontsize=9, color=colors[i] if colors else 'black')
    plt.tight_layout(); plt.show()

print('Готово. Снимков-образцов:', len(SAMPLES))

## 1. Снимок — таблица чисел

In [ ]:
#@title Пиксели одного пятна

from scipy import ndimage as ndi
from skimage import color, measure, morphology

def leaf_and_spots(img, delta=8):
    """Лист — по зелёному оттенку; пятна — то, что краснее листа по оси a* пространства Lab."""
    hsv = color.rgb2hsv(img)
    m = (hsv[..., 0] > 0.10) & (hsv[..., 0] < 0.45) & (hsv[..., 1] > 0.25) & (hsv[..., 2] > 0.25)
    m = morphology.remove_small_objects(ndi.binary_fill_holes(morphology.binary_opening(m, morphology.disk(2))), 500)
    a = color.rgb2lab(img)[..., 1]
    if not m.any():                                   # правило не нашло лист вовсе
        return m, np.zeros_like(m), a
    lab = measure.label(m)
    m = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    m = ndi.binary_fill_holes(morphology.binary_closing(m, morphology.disk(3)))
    lab = measure.label(m)
    m = lab == (np.argmax(np.bincount(lab.ravel())[1:]) + 1)
    inner = morphology.binary_erosion(m, morphology.disk(3))
    sp = inner & (a > np.median(a[inner]) + delta)
    sp = morphology.remove_small_objects(morphology.binary_opening(sp, morphology.disk(1)), 10)
    keep = [r.label for r in measure.regionprops(measure.label(sp)) if r.area >= 12]
    sp = sp & np.isin(measure.label(sp), keep)
    return m, sp, a

def inside_point(mask):
    """Точка глубже всего внутри маски (для клика); пустая маска — центр снимка."""
    if not mask.any():
        return mask.shape[1] // 2, mask.shape[0] // 2
    y, x = np.unravel_index(np.argmax(ndi.distance_transform_edt(mask)), mask.shape)
    return int(x), int(y)

def overlay(img, leaf, spot):
    o = img.astype(float).copy()
    o[~leaf] *= 0.35
    o[spot] = o[spot] * 0.2 + np.array([229, 72, 77]) * 0.8
    return o.astype(np.uint8)

img = sample('apple_rust')
leaf, spot, astar = leaf_and_spots(img)
regs = [r for r in measure.regionprops(measure.label(spot)) if r.area >= 12]
big = max(regs, key=lambda q: q.area)
cy, cx = int(big.centroid[0]), int(big.centroid[1])
print('Размер снимка:', img.shape, '→', img.size, 'чисел')
print('Канал R (красный) в квадрате 6 × 6 вокруг самого большого пятна:')
print(img[cy - 3:cy + 3, cx - 3:cx + 3, 0])
show([img, img[..., 0], img[..., 1], astar], ['снимок', 'R', 'G', 'a* (Lab): зелёный ↔ красный'], cols=4, size=3)

## 2. Задание 1. Классика против SAM
**Классика** — правило без обучения: лист по зелёному оттенку, пятна — то, что краснее листа по оси a*.
**Порог** — насколько краснее.

In [ ]:
#@title Классика за три шага
delta = 8  #@param {type:"slider", min:2, max:20, step:1}
leaf, spot, _ = leaf_and_spots(img, delta)
regs = [r for r in measure.regionprops(measure.label(spot)) if r.area >= 12]
pct = 100 * spot.sum() / leaf.sum()
show([img, leaf, spot, overlay(img, leaf, spot)],
     ['снимок', 'лист', 'пятна', f'пятен {len(regs)} · поражено {pct:.1f} %'], cols=4, size=3)

In [ ]:
#@title То же правило на десяти листьях
TEN = ['apple_rust', 'peach_bact', 'grape_brot', 'corn_gls', 'tom_bact', 'tom_late', 'tom_septoria',
       'tom_target', 'tom_mosaic', 'tom_healthy']
ten = {n: sample(n) for n in TEN}
classic = {}
for n, im in ten.items():
    lf, sp, _ = leaf_and_spots(im, delta)
    classic[n] = (lf, sp, 100 * sp.sum() / max(1, lf.sum()))
show([overlay(ten[n], *classic[n][:2]) for n in TEN],
     [f'{n}\nпоражено {classic[n][2]:.1f} %' if classic[n][0].any() else f'{n}\nлист не найден'
      for n in TEN], cols=5, size=2.8)

**Что заметить.** Где правило ошибается? На многих листьях маска захватила только светлую часть листа — тёмная
половина «не зелёная» по заданному диапазону. Тёмные пятна (виноград) не «краснее» листа — правило их не видит.
Лист во весь кадр или пожелтевший лист сбивают маску листа, а у здорового томата правило не нашло лист вовсе:
его зелёный не попал в заданный диапазон оттенка. Правило годится там, где съёмка однородна.

### SAM: кликнул — получил контур
SAM (Segment Anything) по клику-точке возвращает контур объекта. Здесь — открытая первая версия SAM
(`facebook/sam-vit-base`); в CVAT на стенде — SAM 3, принцип тот же. Кликаем в центр листа — и в центр самого
большого пятна.

In [ ]:
#@title SAM по клику (первый запуск скачивает модель ≈ 375 МБ)
import torch
from transformers import SamModel, SamProcessor
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sam = SamModel.from_pretrained('facebook/sam-vit-base').to(device)
sam_proc = SamProcessor.from_pretrained('facebook/sam-vit-base')

def sam_variants(im, x=None, y=None, box=None):
    """Три варианта маски на одну подсказку — точку (x, y) или рамку box=(x0, y0, x1, y1) — от мелкого к
    крупному, и оценка SAM для каждого."""
    prompt = {'input_boxes': [[list(box)]]} if box is not None else {'input_points': [[[x, y]]]}
    inputs = sam_proc(Image.fromarray(im), return_tensors='pt', **prompt).to(device)
    with torch.no_grad():
        out = sam(**inputs)
    cpu = lambda t: t.cpu() if hasattr(t, 'cpu') else t
    post = getattr(sam_proc, 'post_process_masks', None) or sam_proc.image_processor.post_process_masks
    masks = post(cpu(out.pred_masks), cpu(inputs['original_sizes']), cpu(inputs['reshaped_input_sizes']))[0]
    masks = masks.reshape(-1, *masks.shape[-2:]).cpu().numpy().astype(bool)
    scores = out.iou_scores.reshape(-1).cpu().numpy()
    order = np.argsort([m.sum() for m in masks])          # от мелкого к крупному
    return [masks[i] for i in order], [float(scores[i]) for i in order]

def sam_click(im, x, y):
    """Вариант, в котором SAM уверен больше всего."""
    masks, scores = sam_variants(im, x, y)
    return masks[int(np.argmax(scores))]

rows, pics, titles = [], [], []
for n in TEN:
    im = ten[n]; lf, sp, p_classic = classic[n]
    x, y = inside_point(lf)
    sam_leaf = sam_click(im, x, y)
    iou_leaf = (sam_leaf & lf).sum() / max(1, (sam_leaf | lf).sum())
    p_sam = 100 * (sp & sam_leaf).sum() / max(1, sam_leaf.sum())
    rows.append((n, p_classic, p_sam, iou_leaf))
    o = im.astype(float).copy(); o[~sam_leaf] *= 0.35
    pics.append(o.astype(np.uint8)); titles.append(f'{n}\nлист SAM · IoU с правилом {iou_leaf:.2f}')
show(pics, titles, cols=5, size=2.8)
import pandas as pd
pd.DataFrame(rows, columns=['лист', 'поражено: лист правилом, %', 'поражено: лист SAM, %', 'IoU масок листа']).round(2)

In [ ]:
#@title Пятно: подсказка точкой и подсказка рамкой
name = 'apple_rust'  #@param ["apple_rust", "peach_bact", "grape_brot", "corn_gls", "tom_bact", "tom_late", "tom_septoria", "tom_target", "tom_mosaic"]
pad = 4  #@param {type:"slider", min:0, max:15, step:1}
im = ten[name]; lf, sp, _ = classic[name]
lab = measure.label(sp)
if lab.max() == 0:
    print('Правило не нашло пятен на этом листе — подсказывать некуда. Выберите другой лист.')
else:
    r = max(measure.regionprops(lab), key=lambda q: q.area)
    x, y = inside_point(lab == r.label)
    y0, x0, y1, x1 = r.bbox
    box = (max(0, x0 - pad), max(0, y0 - pad), min(im.shape[1], x1 + pad), min(im.shape[0], y1 + pad))
    pm, ps = sam_variants(im, x, y)
    bm, bs = sam_variants(im, box=box)
    best_box = bm[int(np.argmax(bs))]
    fig, ax = plt.subplots(1, 5, figsize=(17, 4))
    ax[0].imshow(im); ax[0].plot(x, y, 'c*', ms=16)
    ax[0].add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, ec='c', lw=2))
    ax[0].set_title(f'точка и рамка\nправило: пятно {r.area} пикс.', fontsize=10)
    for a, m, sc, t in zip(ax[1:4], pm, ps, ['мелкий', 'средний', 'крупный']):
        o = im.astype(float).copy(); o[m] = o[m] * 0.3 + np.array([229, 72, 77]) * 0.7
        a.imshow(o.astype(np.uint8)); a.set_title(f'точка · {t}\n{m.sum()} пикс. · оценка {sc:.2f}', fontsize=10)
    o = im.astype(float).copy(); o[best_box] = o[best_box] * 0.3 + np.array([229, 72, 77]) * 0.7
    ax[4].imshow(o.astype(np.uint8))
    ax[4].set_title(f'рамка · лучший вариант\n{best_box.sum()} пикс. · оценка {max(bs):.2f}', fontsize=10)
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
    inter = (best_box & (lab == r.label)).sum(); union = (best_box | (lab == r.label)).sum()
    print(f'Точка: варианты {[int(m.sum()) for m in pm]} пикс. — пятно правила {r.area} пикс.')
    print(f'Рамка: {best_box.sum()} пикс.; IoU с пятном правила {inter / max(1, union):.2f}')

**Что заметить.**
1. **Таблица.** Где совпадение масок листа низкое, посмотрите на картинки: чаще прав SAM. Он обводит лист
целиком, а правило по цвету берёт только светлую часть (виноград, септориоз, мишеневидная пятнистость), а у
здорового томата не находит лист вовсе. На фитофторозе бывает наоборот: SAM отрезает поражённый край как
отдельный объект — и доля поражения по его маске выходит меньше.
2. **Пятно.** На одну точку SAM даёт три варианта — и в нашей проверке ни один из них не был пятном: самый мелкий
захватил половину листа, остальные — лист целиком. Маленький объект одной точкой не выделить. Рамка вокруг
пятна помогает: SAM выделил именно пятно — но на 600 пикселей против 241 у правила, вместе с краем и ореолом.
Где кончается пятно, решает инструкция к разметке, а не модель. Подвигайте запас рамки (`pad`) и посмотрите
другие листья. В CVAT мелкие объекты так и размечают: рамкой или несколькими точками, а не одним кликом.
3. SAM находит **контур**, но не знает, **что** это: болезнь, тень или блик. Решение остаётся за правилом,
моделью или человеком — вывод части 3: модель рисует, человек решает.

### Что внутри SAM: тяжёлый глаз один раз, лёгкий карандаш много раз
SAM устроен из двух частей. **Энкодер** — большая сеть — один раз превращает снимок в карту признаков: 256
чисел на каждую клетку сетки 64 × 64. **Декодер** — маленькая сеть — по подсказке (точке, рамке) рисует маску из
этой карты. Меряем время каждой части и смотрим на карту признаков: три главных направления из 256 — как три
цветовых канала.

In [ ]:
#@title Энкодер один раз — декодер на каждый клик
import time
name = 'tom_target'  #@param ["apple_rust", "peach_bact", "grape_brot", "corn_gls", "tom_bact", "tom_late", "tom_septoria", "tom_target", "tom_mosaic"]
im = ten[name]; lf = classic[name][0]
inputs = sam_proc(Image.fromarray(im), return_tensors='pt').to(device)
t0 = time.perf_counter()
with torch.no_grad():
    emb = sam.get_image_embeddings(inputs['pixel_values'])
emb = emb[0] if isinstance(emb, (tuple, list)) else getattr(emb, 'image_embeddings', emb)
t_enc = time.perf_counter() - t0
E = emb[0].cpu().numpy()                                  # 256 × 64 × 64
flat = E.reshape(E.shape[0], -1).T
flat = flat - flat.mean(0)
_, _, vt = np.linalg.svd(flat, full_matrices=False)
rgb = (flat @ vt[:3].T).reshape(E.shape[1], E.shape[2], 3)
rgb = (rgb - rgb.min((0, 1))) / (np.ptp(rgb, (0, 1)) + 1e-9)
h, w = im.shape[:2]
sp = classic[name][1]
lab_sp = measure.label(sp)
if lab_sp.max():                                          # клик 2 — в самое большое пятно правила
    spot_pt = inside_point(lab_sp == max(measure.regionprops(lab_sp), key=lambda q: q.area).label)
else:
    spot_pt = (w // 2, h // 3)
clicks = [inside_point(lf) if lf.any() else (w // 2, h // 2), spot_pt, (8, h - 8)]
post = getattr(sam_proc, 'post_process_masks', None) or sam_proc.image_processor.post_process_masks
masks, t_dec = [], []
for x, y in clicks:
    q = sam_proc(Image.fromarray(im), input_points=[[[x, y]]], return_tensors='pt').to(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = sam(image_embeddings=emb, input_points=q['input_points'], multimask_output=True)
    t_dec.append(time.perf_counter() - t0)
    m = post(out.pred_masks.cpu(), q['original_sizes'].cpu(), q['reshaped_input_sizes'].cpu())[0]
    m = m.reshape(-1, *m.shape[-2:]).numpy().astype(bool)
    masks.append(m[int(out.iou_scores.reshape(-1).argmax())])
fig, ax = plt.subplots(1, 5, figsize=(18, 3.9))
ax[0].imshow(im); ax[0].set_title('1. снимок', fontsize=11)
ax[1].imshow(rgb, interpolation='nearest')
ax[1].set_title(f'2. энкодер: {E.shape[0]} × {E.shape[1]} × {E.shape[2]}\nодин раз · {t_enc:.2f} с', fontsize=11)
for a, (x, y), m, t, where in zip(ax[2:], clicks, masks, t_dec, ['в лист', 'в пятно', 'в фон']):
    o = im.astype(float).copy(); o[m] = o[m] * 0.3 + np.array([72, 161, 250]) * 0.7
    a.imshow(o.astype(np.uint8)); a.plot(x, y, 'r*', ms=16)
    a.set_title(f'3. декодер: клик {where} · {1000 * t:.0f} мс', fontsize=11)
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
print(f'Энкодер: {t_enc:.2f} с на снимок. Декодер: {", ".join(f"{1000 * t:.0f}" for t in t_dec)} мс на клик '
      f'(первый клик дольше — прогрев). Энкодер медленнее декодера в {t_enc / np.median(t_dec):.0f} раз.')
print('Площадь масок, пикс.:', ', '.join(f'клик {w_} — {int(m.sum())}' for w_, m in zip(['в лист', 'в пятно', 'в фон'], masks)))

**Что заметить.** На карте признаков лист и фон уже окрашены по-разному, видны даже жилки — энкодер «разложил»
снимок до всякой подсказки. Поэтому клик стоит доли секунды: декодер только читает готовую карту. В нашей
проверке (процессор, без видеокарты) энкодер работал 7–9 секунд, декодер — около 0,1 секунды на клик. Клик в
пятно даёт крошечную маску, которую почти закрывает звёздочка, или снова лист — смотрите площадь масок под
картинкой: мелкий объект одной точкой выделяется плохо (раздел выше). Клик в фон даёт маску
фона: SAM отвечает на вопрос «где край», а не «что это». На видеокарте всё в десятки раз быстрее, соотношение то
же.

## 3. Свёртка на числах
Свёртка — окно 3 × 3, которое скользит по снимку: умножить числа окна на ядро попарно и сложить. Ядро Собеля
откликается на вертикальные края. Впишите своё ядро — три строки через точку с запятой.

In [ ]:
#@title Окно, ядро и отклик
kernel_text = '-1 0 1; -2 0 2; -1 0 1'  #@param {type:"string"}
kern = np.array([[float(v) for v in row.split()] for row in kernel_text.split(';')])
leaf, spot, _ = leaf_and_spots(img)
g = (color.rgb2gray(img) * 255).astype(int)
sob = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
resp = ndi.correlate(g.astype(float), sob)
inner = morphology.binary_erosion(leaf, morphology.disk(6))
yy, xx = np.unravel_index(np.argmax(np.where(inner, np.abs(resp), 0)), g.shape)
win = g[yy - 1:yy + 2, xx - 1:xx + 2]
print('Окно яркости 3 × 3 на сильном крае листа:'); print(win)
print('Ядро:'); print(kern)
print('Отклик = сумма попарных произведений =', (win * kern).sum())
out = ndi.correlate(g.astype(float), kern)
show([img, np.abs(out)], ['снимок', 'окно прошло по всему снимку'], cols=2, size=4)

**Что сделать.** Поверните ядро (`-1 -2 -1; 0 0 0; 1 2 1`) — какие края теперь видны? Ядро из одинаковых
`1/9` размывает снимок. Сеть подбирает такие ядра сама — тысячи штук — по размеченным примерам.

## 4. IoU и NMS
**IoU** — площадь пересечения рамок, делённая на площадь объединения. **NMS** — из нескольких рамок на один объект
оставить лучшую, убрав те, что сильно с ней пересекаются. Рамка — `x, y, ширина, высота`.

In [ ]:
#@title Калькулятор IoU и NMS
truth_box = '22, 22, 29, 17'  #@param {type:"string"}
model_box = '26, 25, 32, 19'  #@param {type:"string"}
nms_threshold = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

def box(s):
    return [float(v) for v in s.split(',')]

def iou(a, b):
    iw = max(0, min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)

gt, pr = box(truth_box), box(model_box)
print(f'IoU эталона и модели: {iou(gt, pr):.2f}  (часто засчитывают попадание при IoU ≥ 0,5)')

def nms(boxes, scores, thr):
    order = list(np.argsort(scores)[::-1]); keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) <= thr]
    return keep

boxes = [[gt[0] + 1, gt[1], gt[2] + 2, gt[3] + 1], [gt[0] + 4, gt[1] + 3, gt[2] + 2, gt[3] + 1],
         [gt[0] - 3, gt[1] - 2, gt[2] + 1, gt[3] + 3]]
scores = [0.92, 0.81, 0.64]
print('IoU с лучшей рамкой (0,92):', [round(iou(boxes[0], b), 2) for b in boxes[1:]])
print(f'NMS с порогом {nms_threshold}: остались рамки с уверенностью', [scores[i] for i in nms(boxes, scores, nms_threshold)])

**Что сделать.** Сдвиньте рамку модели на 3–4 пикселя — насколько падает IoU у маленького объекта? Поднимите порог
NMS до 0,7: что происходит с дублями?

## 5. Задание 2. Найти провальный класс
Берём сеть ResNet-18, обученную на миллионе обычных фотографий (ImageNet), и учим только **новый последний
слой** на пяти классах томата. Один класс — **редкий**: в обучении у него всего несколько снимков. Меряем
accuracy и macro-F1 и ищем класс, который прячется за средним.

In [ ]:
#@title Собираем данные: 5 классов томата (около минуты)
rare_class = 'Tomato___Target_Spot'  #@param ["Tomato___Target_Spot", "Tomato___Septoria_leaf_spot", "Tomato___Bacterial_spot", "Tomato___Late_blight", "Tomato___healthy"]
rare_train = 5  #@param {type:"slider", min:2, max:100, step:1}
CLASSES = ['Tomato___healthy', 'Tomato___Bacterial_spot', 'Tomato___Late_blight', 'Tomato___Septoria_leaf_spot',
           'Tomato___Target_Spot']
RU = {'Tomato___healthy': 'здоровый', 'Tomato___Bacterial_spot': 'бактериальная пятнистость',
      'Tomato___Late_blight': 'фитофтороз', 'Tomato___Septoria_leaf_spot': 'септориоз',
      'Tomato___Target_Spot': 'мишеневидная пятнистость'}
ALL = list_dataset()
rng = np.random.default_rng(2026 + participant)
tr_paths, tr_y, te_paths, te_y = [], [], [], []
for c_i, c in enumerate(CLASSES):
    ps = sorted(p for p in ALL if cls(p) == c)
    ps = [ps[i] for i in rng.permutation(len(ps))]
    n_tr, n_te = (rare_train, 15) if c == rare_class else (100, 50)
    tr_paths += ps[:n_tr]; tr_y += [c_i] * n_tr
    te_paths += ps[n_tr:n_tr + n_te]; te_y += [c_i] * n_te
tr_imgs, te_imgs = get_many(tr_paths), get_many(te_paths)
tr_y, te_y = np.array(tr_y), np.array(te_y)
for c_i, c in enumerate(CLASSES):
    print(f'{RU[c]:<26} обучение {(tr_y == c_i).sum():>3} · проверка {(te_y == c_i).sum():>3}')

In [ ]:
#@title Признаки готовой сети + новый последний слой
import torch, torchvision
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
device = 'cuda' if torch.cuda.is_available() else 'cpu'
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
net = torchvision.models.resnet18(weights=weights)
net.fc = torch.nn.Identity()
net = net.eval().to(device)
prep = weights.transforms()

def features(images, bs=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(images), bs):
            x = torch.stack([prep(Image.fromarray(im)) for im in images[i:i + bs]]).to(device)
            out.append(net(x).cpu().numpy())
    return np.concatenate(out)

Ftr, Fte = features(tr_imgs), features(te_imgs)
head = LogisticRegression(max_iter=3000).fit(Ftr, tr_y)
pred = head.predict(Fte)
dummy = DummyClassifier(strategy='most_frequent').fit(Ftr, tr_y).predict(Fte)
print(f'«Всегда самый частый класс»: accuracy {accuracy_score(te_y, dummy):.1%} · '
      f'macro-F1 {f1_score(te_y, dummy, average="macro"):.2f}')
print(f'Готовая сеть + новый слой:   accuracy {accuracy_score(te_y, pred):.1%} · '
      f'macro-F1 {f1_score(te_y, pred, average="macro"):.2f}\n')
print(classification_report(te_y, pred, target_names=[RU[c] for c in CLASSES], digits=2, zero_division=0))
ConfusionMatrixDisplay.from_predictions(te_y, pred, display_labels=[RU[c][:14] for c in CLASSES],
                                        xticks_rotation=30, cmap='Blues')
plt.title('строки — правда, столбцы — ответ модели'); plt.show()

**В нашем прогоне** (номер участника 0, проверка ноутбука, октябрь 2026): accuracy 89,8 %, macro-F1 0,80. У четырёх классов F1
0,90–0,94, у мишеневидной пятнистости — 0,33: модель узнала её на 3 листьях из 15, а 9 назвала **здоровыми**.
Больной лист пропущен — самая дорогая ошибка, а общая точность 90 % её не показывает. У вас числа могут слегка
отличаться.

**Что сделать.**
1. У какого класса F1 ниже всех? С какими классами модель его путает (матрица ошибок)?
2. Сравните accuracy и macro-F1: какая цифра честнее говорит о редком классе?
3. Увеличьте `rare_train` до 20, 50, 100 и перезапустите обе ячейки: сколько снимков нужно редкому классу?
4. Сделайте редким другой класс. Все ли классы одинаково «страдают» от нехватки примеров?

### Один лист через сеть
Что происходит внутри, пока сеть отвечает «септориоз, 0,9»? Пропускаем один лист из проверки и смотрим, что
выходит после каждого блока ResNet-18: сначала карты краёв и пятен, потом всё более крупные и абстрактные, в
конце — 512 чисел, из которых новый слой собирает ответ.

In [ ]:
#@title Лист → слои сети → вектор → ответ
leaf_class = 'Tomato___Septoria_leaf_spot'  #@param ["Tomato___healthy", "Tomato___Bacterial_spot", "Tomato___Late_blight", "Tomato___Septoria_leaf_spot", "Tomato___Target_Spot"]
leaf_index = 0  #@param {type:"slider", min:0, max:14, step:1}
c_i = CLASSES.index(leaf_class)
k = np.nonzero(te_y == c_i)[0][leaf_index]
im = te_imgs[k]
acts = {}
names = ['conv1', 'layer2', 'layer4']
hooks = [getattr(net, n).register_forward_hook(lambda m, i, o, n=n: acts.__setitem__(n, o[0].cpu().numpy()))
         for n in names]
with torch.no_grad():
    vec = net(prep(Image.fromarray(im)).unsqueeze(0).to(device)).cpu().numpy()
for hk in hooks: hk.remove()
prob = head.predict_proba(vec)[0]
fig, ax = plt.subplots(1, 6, figsize=(18, 3.6), gridspec_kw={'width_ratios': [1, 1, 1, 1, 0.5, 1.3]})
ax[0].imshow(im); ax[0].set_title(f'1. снимок\nправда: {RU[leaf_class]}', fontsize=11)
for j, n in enumerate(names):
    a_ = acts[n]; ch = int(a_.mean((1, 2)).argmax())
    ax[j + 1].imshow(a_[ch], cmap='magma')
    ax[j + 1].set_title(f'{j + 2}. {n}: {a_.shape[0]} карт {a_.shape[1]}×{a_.shape[2]}\nсамая активная — №{ch}', fontsize=11)
ax[4].imshow(vec.reshape(32, 16), cmap='Blues')
ax[4].set_title('5. вектор\n512 чисел', fontsize=11)
order = np.argsort(prob)
short = {c: RU[c].replace(' пятнистость', ' пятн.') for c in CLASSES}
ax[5].barh([short[CLASSES[i]] for i in order], prob[order],
           color=['#48A1FA' if i == c_i else '#C2DFFD' for i in order])
for y_, i in enumerate(order):
    ax[5].text(prob[i] + 0.01, y_, f'{prob[i]:.2f}', va='center', fontsize=10)
ax[5].set_xlim(0, 1.15); ax[5].set_title('6. новый слой: вероятности', fontsize=11)
for a in ax[:5]: a.axis('off')
fig.subplots_adjust(left=0.01, right=0.97, wspace=0.12); plt.show()
print(f'Ответ: {RU[CLASSES[prob.argmax()]]}, {prob.max():.2f} · правда: {RU[leaf_class]}')

**Что заметить.** Первые карты похожи на снимок: края, жилки, пятна — это фильтры из раздела 3. Чем глубже, тем
меньше карта и тем меньше она похожа на картинку: 7 × 7 клеток, зато 512 разных «вопросов» к снимку. Ответ
собирает только последний, новый слой — из 512 чисел. Поменяйте `leaf_index` и `leaf_class`: найдите лист, на
котором сеть ошибается, и посмотрите, с каким классом она его путает.

### Как учится новый слой: ошибка → поправка → снова
Выше новый слой обучен готовой функцией. Здесь — то же самое шагами: показали все снимки обучения, посчитали
ошибку, чуть сдвинули веса, повторили. Следим за общей ошибкой и за тем же листом.

In [ ]:
#@title Обучение по шагам: общая ошибка и один лист
steps = 100  #@param {type:"slider", min:10, max:600, step:10}
lr = 0.001  #@param [0.0003, 0.001, 0.003, 0.01, 0.1] {type:"raw"}
torch.manual_seed(participant)
mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
Xtr = torch.tensor((Ftr - mu) / sd, dtype=torch.float32); Ytr = torch.tensor(tr_y)
xv = torch.tensor((vec - mu) / sd, dtype=torch.float32)
lin = torch.nn.Linear(Xtr.shape[1], len(CLASSES))
opt = torch.optim.Adam(lin.parameters(), lr=lr)
losses, p_true, p_top = [], [], []
for t in range(steps + 1):
    with torch.no_grad():
        pv = torch.softmax(lin(xv), 1)[0]
    p_true.append(float(pv[c_i])); p_top.append(int(pv.argmax()))
    loss = torch.nn.functional.cross_entropy(lin(Xtr), Ytr)
    losses.append(float(loss))
    if t < steps:
        opt.zero_grad(); loss.backward(); opt.step()
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].plot(losses, color='#012548', lw=2.5); ax[0].set_xlabel('шаг'); ax[0].set_ylabel('ошибка на обучении')
ax[0].set_title('общая ошибка падает', fontsize=12)
ax[1].plot(p_true, color='#48A1FA', lw=2.5); ax[1].axhline(1 / len(CLASSES), color='gray', ls='--')
ax[1].set_ylim(0, 1); ax[1].set_xlabel('шаг'); ax[1].set_ylabel('вероятность правильного класса')
ax[1].set_title(f'тот же лист: {RU[leaf_class]}', fontsize=12)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
first = next((t for t, c in enumerate(p_top) if c == c_i), None)
print(f'Шаг 0: «{RU[CLASSES[p_top[0]]]}», {p_true[0]:.2f} за правильный класс — веса случайные.')
print(f'Шаг {steps}: «{RU[CLASSES[p_top[-1]]]}», {p_true[-1]:.2f} за правильный класс.')
print(f'Впервые ответил правильно на шаге {first}.' if first is not None else 'Правильно так и не ответил.')

**Что заметить.** На шаге 0 веса случайные — ответ наугад: в нашей проверке 0,12 за правильный класс, а
правильно сеть ответила впервые на шаге 11. Каждый шаг
чуть сдвигает веса туда, где ошибка меньше, — и вероятность правильного класса у нашего листа растёт. `lr` —
размер шага: поднимите до 0,01 — доучится за десяток шагов; опустите до 0,0003 — за 100 шагов не доучится.
Ошибка на обучении уходит почти в ноль, но это не точность на новых листьях: её меряли на проверке выше.

### Уверенность против правоты
Модель говорит «0,9». Права ли она в 90 % таких случаев? Делим ответы на проверке по уверенности и считаем, как
часто модель права в каждой группе.

In [ ]:
#@title Уверенность против правоты
P = head.predict_proba(Fte)
conf, right = P.max(1), P.argmax(1) == te_y
edges = np.array([0.2, 0.5, 0.7, 0.8, 0.9, 0.95, 1.0001])
rows, ece = [], 0.0
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = (conf >= lo) & (conf < hi)
    if sel.sum():
        rows.append((f'{lo:.2f}–{min(hi, 1):.2f}', int(sel.sum()), conf[sel].mean(), right[sel].mean()))
        ece += sel.mean() * abs(conf[sel].mean() - right[sel].mean())
import pandas as pd
tab = pd.DataFrame(rows, columns=['уверенность', 'ответов', 'средняя уверенность', 'доля верных'])
plt.figure(figsize=(6, 4.5))
plt.plot([0, 1], [0, 1], '--', color='gray', label='идеально честная модель')
plt.plot(tab['средняя уверенность'], tab['доля верных'], 'o-', color='#48A1FA', lw=2.5, ms=8, label='наша модель')
for _, r in tab.iterrows():
    plt.text(r['средняя уверенность'], r['доля верных'] - 0.06, str(r['ответов']), ha='center', fontsize=9)
plt.xlabel('что говорит модель (уверенность)'); plt.ylabel('как часто права'); plt.xlim(0.2, 1.02); plt.ylim(0, 1.05)
plt.legend(loc='upper left'); plt.grid(alpha=0.3); plt.show()
print(f'ECE — средний разрыв между уверенностью и правотой: {ece:.3f}')
tab.round(2)

**Что заметить.** Точки под диагональю — модель самоуверенна: говорит больше, чем знает. Над диагональю —
скромничает. Числа у точек — сколько ответов в группе: где их мало, вывод шаткий. Сделайте редкий класс ещё
реже (`rare_train` = 2) и перезапустите: где окажутся точки?

## 6. Задание 3. Active learning своими руками
Разметка дорогая. Модель учится на 20 размеченных цифрах и **сама выбирает**, какие размечать дальше: те, в
которых она меньше всего уверена. Сравниваем со случайным выбором.

In [ ]:
#@title Случайный отбор против «сначала неуверенные»
batch = 10  #@param {type:"slider", min:1, max:50, step:1}
repeats = 5  #@param {type:"slider", min:1, max:10, step:1}
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X, y = load_digits(return_X_y=True); X = X / 16.0
Xp, Xt, yp, yt = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
budgets = [20, 40, 60, 80, 100, 150, 200, 300, 400]

def run(strategy, seed):
    rr = np.random.default_rng(seed)
    lab = list(rr.choice(len(Xp), 20, replace=False)); accs = []
    for b in budgets:
        while len(lab) < b:
            clf = LogisticRegression(max_iter=2000).fit(Xp[lab], yp[lab])
            pool = np.setdiff1d(np.arange(len(Xp)), lab)
            take = min(batch, b - len(lab))
            if strategy == 'random':
                pick = rr.choice(pool, take, replace=False)
            else:
                pr = np.sort(clf.predict_proba(Xp[pool]), 1)
                pick = pool[np.argsort(pr[:, -1] - pr[:, -2])[:take]]
            lab += list(pick)
        accs.append((LogisticRegression(max_iter=2000).fit(Xp[lab], yp[lab]).predict(Xt) == yt).mean())
    return accs

seeds = range(100 * participant, 100 * participant + repeats)
R = np.mean([run('random', s) for s in seeds], 0)
A = np.mean([run('active', s) for s in seeds], 0)
plt.figure(figsize=(8, 4))
plt.plot(budgets, R * 100, 'o-', color='gray', label='случайный отбор')
plt.plot(budgets, A * 100, 'o-', color='#48A1FA', lw=3, label='сначала неуверенные')
plt.xlabel('сколько изображений разметили'); plt.ylabel('точность, %'); plt.legend(); plt.grid(alpha=0.3); plt.show()
for b, r_, a_ in zip(budgets, R, A):
    print(f'{b:>4} размечено: случайно {r_:.1%} · active learning {a_:.1%}')

**Что сделать.** Сколько разметки экономит active learning, чтобы дойти до 91 %? Увеличьте `batch` до 50: модель
выбирает реже и «пачками» — что стало с выигрышем? Так и на практике: чем реже переобучаемся, тем меньше пользы
от умного выбора.

---
*© 2026 Наталия Тылевич. Авторский учебный материал блока «Анализ изображений: от данных к модели». Можно выполнять и хранить личную копию для заданий курса; распространять и использовать в других курсах — только с согласия автора. Условия — `LICENSE.md` в репозитории. Если ссылка не активна — обратитесь к автору.*